# 🚀 Notebook 3 — Deploy Claire to AgentCore

In Notebooks 1-2 you built the agent and tested it locally.
Now deploy it to **Amazon Bedrock AgentCore** — managed infrastructure,
session isolation, observability, zero orchestration code.

You'll:
1. Write the agent entry point
2. Configure `.bedrock_agentcore.yaml`
3. Deploy with the AgentCore CLI
4. Invoke — same PA queries, now production-grade

---
## 3.1 — Create Project Directory

In [ ]:
import os
PROJECT = os.path.expanduser('~/SageMaker/claire-agentcore')
os.makedirs(PROJECT, exist_ok=True)
print(f'Project dir: {PROJECT}')

---
## 3.2 — Write the Agent

Same code from Notebooks 1-2, wrapped in `BedrockAgentCoreApp`.

In [ ]:
!pip install --quiet python-dotenv 2>/dev/null


In [ ]:
%%writefile ~/SageMaker/claire-agentcore/agent.py
#!/usr/bin/env python3
"""Claire HCLS Agent — AgentCore deployment"""
import os
from dotenv import load_dotenv
load_dotenv()

from strands import Agent
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters
from strands.models.bedrock import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

model = BedrockModel(model_id='us.anthropic.claude-sonnet-4-5-20250929-v1:0', streaming=False)

server_params = StdioServerParameters(
    command='uvx',
    args=['teradata-mcp-server'],
    env={
        'DATABASE_URI': os.getenv('TERADATA_DATABASE_URI', ''),
        'LOGMECH': os.getenv('LOGMECH', 'TD2')
    }
)
teradata_tool = MCPClient(lambda: stdio_client(server_params), startup_timeout=300)

app = BedrockAgentCoreApp()

SYSTEM_PROMPT = '''You are Claire, a healthcare claims and prior authorization AI agent.
You have access to a Teradata HCLS database via MCP tools.

Key tables: prior_auth_request, member, provider, facility, auth_cpt_code, cpt_codes,
auth_icd_code, auth_clinical_notes, auth_status, care_pathways,
WORKERS_COMP_HDR, WORKERS_COMP_DTL, alerts.

CRITICAL: Use ordering_provider_id (NOT provider_id). Use primary_specialty (NOT specialty).
Use Teradata SQL (TOP instead of LIMIT). alerts uses object_id for provider NPI.
'''

@app.entrypoint
def invoke(payload):
    agent = Agent(model=model, tools=[teradata_tool], system_prompt=SYSTEM_PROMPT)
    user_message = payload.get('prompt', 'What tables are available?')
    result = agent(user_message)
    return {'response': result.message}

if __name__ == '__main__':
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == '--interactive':
        agent = Agent(model=model, tools=[teradata_tool], system_prompt=SYSTEM_PROMPT)
        while True:
            q = input('You: ').strip()
            if q.lower() in ['quit', 'exit', 'q']: break
            if q: print(f'Claire: {agent(q).message}\n')
    else:
        app.run()

---
## 3.3 — Write the .env

In [ ]:
# Copy .env from workshop dir (credentials stay in .env only, never in code)
!cp ~/SageMaker/claire-workshop/.env ~/SageMaker/claire-agentcore/.env
!echo 'AWS_DEFAULT_REGION=us-east-1' >> ~/SageMaker/claire-agentcore/.env
!echo '.env copied. Contents:' && cat ~/SageMaker/claire-agentcore/.env | sed 's/password=.*/password=****/'

The `.env` file lives in your workshop directory and is **never committed to code**.
In production you'd use AWS Secrets Manager or AgentCore credential management.

---
## 3.4 — Write the AgentCore Config

In [ ]:
import os
ACCOUNT = !aws sts get-caller-identity --query Account --output text
ACCOUNT_ID = ACCOUNT[0].strip()
PROJECT = os.path.expanduser('~/SageMaker/claire-agentcore')

config = f'''default_agent: agent
agents:
  agent:
    name: clairehcls
    entrypoint: {PROJECT}/agent.py
    deployment_type: container
    platform: linux/arm64
    source_path: {PROJECT}
    aws:
      execution_role_auto_create: true
      ecr_auto_create: true
      account: "{ACCOUNT_ID}"
      region: us-east-1
      network_configuration:
        network_mode: PUBLIC
      observability:
        enabled: true
    memory:
      mode: NO_MEMORY
'''

with open(f'{PROJECT}/.bedrock_agentcore.yaml', 'w') as f:
    f.write(config)

print(f'Account: {ACCOUNT_ID}')
print(f'Config written to {PROJECT}/.bedrock_agentcore.yaml')

---
## 3.5 — Write the Dockerfile

In [ ]:
# Read creds from .env and bake into Dockerfile
# (The .dockerignore excludes .env files, so we set ENV directly)
import os
from dotenv import load_dotenv
load_dotenv(os.path.expanduser('~/SageMaker/claire-workshop/.env'))
td_uri = os.getenv('TERADATA_DATABASE_URI', '')

dockerfile = f'''FROM --platform=linux/arm64 ghcr.io/astral-sh/uv:python3.12-bookworm-slim
WORKDIR /app
RUN apt-get update && apt-get install -y gcc && rm -rf /var/lib/apt/lists/*
COPY requirements.txt requirements.txt
RUN pip install --no-cache-dir -r requirements.txt
RUN pip install aws-opentelemetry-distro==0.12.2
COPY . .
ENV PYTHONUNBUFFERED=1
ENV AWS_REGION=us-east-1
ENV TERADATA_DATABASE_URI={td_uri}
ENV LOGMECH=TD2
EXPOSE 8080
CMD ["opentelemetry-instrument", "python", "agent.py"]
'''

with open(os.path.expanduser('~/SageMaker/claire-agentcore/Dockerfile'), 'w') as f:
    f.write(dockerfile)

print('Dockerfile written with TERADATA_DATABASE_URI baked in.')
print(f'URI host: {td_uri.split("@")[1] if "@" in td_uri else "(check .env)"}')

In [ ]:
%%writefile ~/SageMaker/claire-agentcore/requirements.txt
strands-agents>=1.19.0
mcp>=1.22.0
boto3
bedrock-agentcore>=1.1.1
python-dotenv
teradata-mcp-server>=0.1.0

---
## 3.6 — Deploy

This builds a container, pushes to ECR, and deploys to AgentCore. Takes ~3-5 min.

In [ ]:
import subprocess, json, boto3

PROJECT = '~/SageMaker/claire-agentcore'

def patch_agentcore_roles():
    """Ensure AgentCore auto-created roles have correct permissions."""
    policy = json.dumps({'Version': '2012-10-17', 'Statement': [{
        'Effect': 'Allow',
        'Action': ['ecr:*', 'bedrock:*', 'bedrock-agentcore:*', 'logs:*', 's3:*', 'sts:*'],
        'Resource': '*'
    }]})
    iam = boto3.client('iam')
    for role in iam.list_roles(MaxItems=100)['Roles']:
        if 'AgentCoreSDK' in role['RoleName']:
            iam.put_role_policy(RoleName=role['RoleName'], PolicyName='WorkshopFullAccess', PolicyDocument=policy)

# Deploy (first run creates roles, may fail)
print('Deploying to AgentCore...')
!cd {PROJECT} && agentcore deploy --auto-update-on-conflict 2>&1 | tail -5

# Patch roles and retry if needed
patch_agentcore_roles()
!cd {PROJECT} && agentcore deploy --auto-update-on-conflict 2>&1 | tail -10

---
## 3.7 — Watch the Agent Think (Live Logs)

The invoke call takes 15-60 seconds. While it runs, you can watch the agent's
reasoning, SQL queries, and MCP tool calls in real-time from a second terminal.

### How to open a second terminal

**Classic Jupyter:**
1. Go back to the Jupyter file browser tab in your browser
2. Click **New ▼** (top right) → **Terminal**
3. A new terminal tab opens

**JupyterLab:**
1. Click **File** → **New** → **Terminal**
2. Or click the **+** tab → **Terminal**

### Run this in the second terminal to stream logs

Run the cell below first — it prints the exact command to paste into your second terminal.

In [ ]:
# Print the exact log-tail command for your deployed agent
import subprocess
try:
    status = subprocess.check_output(
        'cd ~/SageMaker/claire-agentcore && agentcore status 2>/dev/null | grep -i runtime',
        shell=True
    ).decode().strip()
    print('Agent status:', status)
except:
    pass

# Find the log group
import boto3
logs = boto3.client('logs')
groups = logs.describe_log_groups(logGroupNamePrefix='/aws/bedrock-agentcore/runtimes/clairehcls')['logGroups']
if groups:
    lg = groups[0]['logGroupName']
    print(f'\n📋 Copy this command into your second terminal:\n')
    print(f'aws logs tail "{lg}" --follow')
    print(f'\nThis shows the agent\'s reasoning, SQL queries, and MCP calls in real-time.')
else:
    print('No log group found yet. Deploy the agent first (cell above), then re-run this cell.')

---
## 3.8 — Invoke the Deployed Agent

Start the log tail in your second terminal, then run the invoke below.
Watch the logs while it processes — you'll see the agent think, write SQL, and query Teradata.

In [ ]:
import json

# CLI invoke (uncomment after deploying)
# !cd ~/SageMaker/claire-agentcore && agentcore invoke '{json.dumps({"prompt": "Show me the dashboard for PA-2026-003417"})}'

In [ ]:
# Check status
# !cd ~/SageMaker/claire-agentcore && agentcore status

---
## 3.9 — What Changed?

| Component | Notebooks 1-2 | AgentCore |
|---|---|---|
| Agent loop | Strands SDK (local) | Managed runtime |
| MCP connection | MCPClient (local process) | Container with MCP baked in |
| Auth | IAM role on SageMaker | IAM role on Fargate |
| Session isolation | None (single notebook) | microVM per session |
| Observability | print() | CloudWatch traces |
| Scaling | 1 user | Auto-scaling |
| What you wrote | agent.py + skills | agent.py + skills (same file) |

The agent code is **identical**. Only the infrastructure changed.

---

**➡ Continue to [Notebook 4 — Production Patterns](04-production-patterns.ipynb)**

---
## Clean Up

In [ ]:
# Uncomment to destroy:
# !cd ~/SageMaker/claire-agentcore && agentcore destroy